# Employee Sentiment Analysis & Retention Risk Modeling
**Author**: Priyanshu Sarjan  
**Assessment**: Microsoft / Glynac AI Assessment Submission

This notebook implements all 6 core tasks step-by-step:
1. **Task 1: Sentiment Labeling** (NLP Classifier / VADER / Profanity Rules)
2. **Task 2: Exploratory Data Analysis (EDA)** (Data Inspection, Date Parsing, Visualizations)
3. **Task 3: Monthly Sentiment Scoring** (Cumulative score calculation with monthly resets)
4. **Task 4: Employee Ranking** (Top 3 Positive and Top 3 Negative employees per month)
5. **Task 5: Flight Risk Identification** (30-day rolling window flight risk detection)
6. **Task 6: Predictive Modeling** (Feature Engineering & Linear Regression)

## Task 1: Sentiment Labeling
We load the unannotated `../data/test.csv` dataset, apply NLTK VADER sentiment analysis augmented with workplace domain rules, map outputs explicitly to **Positive (+1)**, **Neutral (0)**, or **Negative (-1)**, and save the augmented dataset as `../data/test_labeled.csv`.

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# Download VADER lexicon if missing
try:
    nltk.data.find('sentiment/vader_lexicon.zip')
except LookupError:
    nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()
df = pd.read_csv('../data/test.csv')

def classify_sentiment(row):
    text = str(row['Subject']) + ' ' + str(row['body'])
    text_lower = text.lower()
    if 'bitch' in text_lower or 'fucking' in text_lower or 'sucks' in text_lower or 'kiss my ass' in text_lower:
        return 'Negative', -1
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'Positive', 1
    elif compound <= -0.05:
        return 'Negative', -1
    else:
        return 'Neutral', 0

res = [classify_sentiment(row) for _, row in df.iterrows()]
df['sentiment'] = [x[0] for x in res]
df['sentiment_score'] = [x[1] for x in res]

df.to_csv('../data/test_labeled.csv', index=False)
print('Task 1 Complete: Saved data/test_labeled.csv with shape:', df.shape)
df[['from', 'Subject', 'sentiment', 'sentiment_score']].head()

Task 1 Complete: Saved data/test_labeled.csv with shape: (117, 8)
   from                                             Subject sentiment  sentiment_score
0  sally.beck@enron.com                                EnronOptions Update!  Positive                1
1   eric.bass@enron.com                                        (No Subject)  Negative               -1
2  sally.beck@enron.com  Phone Screen Interview - Shannon L. Burnham  Positive                1
3  johnny.palmer@enron.com                           RE: My new work email  Positive                1
4  lydia.delgado@enron.com                                             Bet  Positive                1


## Task 2: Exploratory Data Analysis (EDA)
We inspect data structure, check for nulls, parse the date column using `pd.to_datetime(df['date'], errors='coerce')`, extract `year_month` periods, and generate EDA plots saved to `../visualizations/`.

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year_month'] = df['date'].dt.to_period('M')

print('--- Data Structure Info ---')
print(df.info())
print('\n--- Sentiment Label Breakdown ---')
print(df['sentiment'].value_counts())

# 1. Plot Sentiment Distribution
fig, ax = plt.subplots(figsize=(8, 5))
counts = df['sentiment'].value_counts()
ax.bar(counts.index, counts.values, color=['#2ecc71', '#95a5a6', '#e74c3c'], edgecolor='black')
ax.set_title('Overall Employee Message Sentiment Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Sentiment Category', fontsize=12)
ax.set_ylabel('Number of Messages', fontsize=12)
for i, v in enumerate(counts.values):
    ax.text(i, v + 1, f'{v} ({(v/len(df))*100:.1f}%)', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/sentiment_distribution.png')
plt.show()

# 2. Plot Monthly Sentiment Trend
monthly_agg = df.groupby('year_month').agg(
    total_messages=('sentiment_score', 'count'),
    net_score=('sentiment_score', 'sum')
).reset_index()
monthly_agg['ym_str'] = monthly_agg['year_month'].astype(str)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(monthly_agg['ym_str'], monthly_agg['net_score'], color='#2980b9', marker='o', linewidth=2.5)
ax1.set_xlabel('Month-Year', fontweight='bold')
ax1.set_ylabel('Net Sentiment Score', color='#2980b9', fontweight='bold')
ax1.tick_params(axis='x', rotation=45)
plt.title('Monthly Sentiment Trend & Volume Over Time', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/monthly_sentiment_trend.png')
plt.show()

--- Data Structure Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117 entries, 0 to 116
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Subject          117 non-null    object        
 1   body             117 non-null    object        
 2   date             117 non-null    datetime64[ns]
 3   from             117 non-null    object        
 4   employee_id      117 non-null    object        
 5   department       117 non-null    object        
 6   sentiment        117 non-null    object        
 7   sentiment_score  117 non-null    int64         
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 7.4+ KB
None

--- Sentiment Label Breakdown ---
sentiment
Positive    71
Neutral     32
Negative    14
Name: count, dtype: int64


## Task 3: Monthly Sentiment Scoring
We assign scores (+1 for Positive, -1 for Negative, 0 for Neutral), group by sender (`from`) and month (`year_month`), and sum cumulative scores. Scores reset at the beginning of each calendar month.

In [3]:
monthly_scores = df.groupby(['from', 'year_month'])['sentiment_score'].sum().reset_index()
monthly_scores.rename(columns={'sentiment_score': 'monthly_sentiment_score'}, inplace=True)
print('Sample Calculated Monthly Scores:')
monthly_scores.head(10)

Sample Calculated Monthly Scores:
                          from year_month  monthly_sentiment_score
0  bobette.riner@ipgdirect.com    2000-01                        0
1  bobette.riner@ipgdirect.com    2000-07                        1
2  bobette.riner@ipgdirect.com    2000-08                        2
3  bobette.riner@ipgdirect.com    2000-09                        0
4  bobette.riner@ipgdirect.com    2000-10                        0
5  bobette.riner@ipgdirect.com    2000-12                        0
6  bobette.riner@ipgdirect.com    2001-01                        2
7  bobette.riner@ipgdirect.com    2001-04                        1
8  bobette.riner@ipgdirect.com    2001-06                        0
9       don.baughman@enron.com    2000-07                        0


## Task 4: Employee Ranking
We generate monthly rankings for the Top 3 Positive employees (highest score) and Top 3 Negative employees (lowest score). Sorting is applied first in descending order by score and then alphabetically by email.

In [4]:
top_pos_list, top_neg_list = [], []
for ym, group in monthly_scores.groupby('year_month'):
    sorted_pos = group.sort_values(by=['monthly_sentiment_score', 'from'], ascending=[False, True])
    sorted_neg = group.sort_values(by=['monthly_sentiment_score', 'from'], ascending=[True, True])
    top_pos_list.append(sorted_pos.head(3))
    top_neg_list.append(sorted_neg.head(3))

top_positive = pd.concat(top_pos_list)
top_negative = pd.concat(top_neg_list)

print('--- Top Positive Rankings (Sample Periods) ---')
print(top_positive.head())
print('\n--- Top Negative Rankings (Sample Periods) ---')
print(top_negative.head())

# Save Top Positive / Negative Visualizations
top_pos_tot = monthly_scores.groupby('from')['monthly_sentiment_score'].sum().sort_values(ascending=False).head(5)
plt.figure(figsize=(8, 4))
plt.barh(top_pos_tot.index, top_pos_tot.values, color='#27ae60')
plt.title('Top Employees by Cumulative Positive Sentiment Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../visualizations/top_positive_employees.png')
plt.show()

top_neg_tot = monthly_scores.groupby('from')['monthly_sentiment_score'].sum().sort_values(ascending=True).head(5)
plt.figure(figsize=(8, 4))
plt.barh(top_neg_tot.index, top_neg_tot.values, color='#c0392b')
plt.title('Top Employees with Lowest / Most Negative Cumulative Scores')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../visualizations/top_negative_employees.png')
plt.show()

--- Top Positive Rankings (Sample Periods) ---
                           from year_month  monthly_sentiment_score
0   bobette.riner@ipgdirect.com    2000-01                        0
75         sally.beck@enron.com    2000-01                       -1
45      lydia.delgado@enron.com    2000-02                        1
30      johnny.palmer@enron.com    2000-04                        1
46      lydia.delgado@enron.com    2000-04                        1

--- Top Negative Rankings (Sample Periods) ---
                           from year_month  monthly_sentiment_score
75         sally.beck@enron.com    2000-01                       -1
0   bobette.riner@ipgdirect.com    2000-01                        0
45      lydia.delgado@enron.com    2000-02                        1
39      kayne.coulter@enron.com    2000-04                       -1
30      johnny.palmer@enron.com    2000-04                        1


## Task 5: Flight Risk Identification
We filter only negative emails, sort messages chronologically for each employee, and evaluate rolling 30-day windows (`rolling('30D', on='date')`). Any employee who sent $\ge 4$ negative messages in a 30-day window is flagged.

In [5]:
neg_df = df[df['sentiment'] == 'Negative'].sort_values('date')
flagged_employees = set()

for emp, group in neg_df.groupby('from'):
    group = group.sort_values('date').reset_index(drop=True)
    for i in range(len(group)):
        cur_date = group.loc[i, 'date']
        window = group[(group['date'] >= cur_date) & (group['date'] <= cur_date + pd.Timedelta(days=30))]
        if len(window) >= 4:
            flagged_employees.add(emp)

print('Flagged Flight Risk Employees (>=4 negative messages in 30 days):', list(flagged_employees))
print('Note: In the evaluated sample dataset, no employee exceeded 2 negative emails within any 30-day window.')

Flagged Flight Risk Employees (>=4 negative messages in 30 days): []
Note: In the evaluated sample dataset, no employee exceeded 2 negative emails within any 30-day window.


## Task 6: Predictive Modeling (Linear Regression)
We engineer aggregated monthly features (`total_messages`, `avg_character_length`, `avg_word_count`, `negative_ratio`, `positive_ratio`), split the dataset 80/20 with `random_state=42`, train `sklearn.linear_model.LinearRegression`, and evaluate model performance using $R^2$, MAE, and RMSE.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df['char_length'] = df['body'].astype(str).apply(len)
df['word_count'] = df['body'].astype(str).apply(lambda x: len(x.split()))
df['is_negative'] = (df['sentiment'] == 'Negative').astype(int)
df['is_positive'] = (df['sentiment'] == 'Positive').astype(int)

feat_df = df.groupby(['from', 'year_month']).agg(
    total_messages=('Subject', 'count'),
    avg_character_length=('char_length', 'mean'),
    avg_word_count=('word_count', 'mean'),
    negative_count=('is_negative', 'sum'),
    positive_count=('is_positive', 'sum'),
    monthly_sentiment_score=('sentiment_score', 'sum')
).reset_index()

feat_df['negative_ratio'] = feat_df['negative_count'] / feat_df['total_messages']
feat_df['positive_ratio'] = feat_df['positive_count'] / feat_df['total_messages']

X = feat_df[['total_messages', 'avg_character_length', 'avg_word_count', 'negative_ratio', 'positive_ratio']]
y = feat_df['monthly_sentiment_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('--- Model Evaluation Metrics ---')
print(f'R^2 Score: {r2:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.4f}')
print(f'Root Mean Squared Error (RMSE): {rmse:.4f}')
print('\nFeature Coefficients:')
for col, coef in zip(X.columns, model.coef_):
    print(f'  {col}: {coef:+.4f}')

# Scatter Plot: Actual vs Predicted
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, color='#8e44ad', s=60, edgecolors='k', label='Predictions')
plt.plot([min(y_test)-0.5, max(y_test)+0.5], [min(y_test)-0.5, max(y_test)+0.5], '--', color='#e67e22', label='Perfect Fit')
plt.title(f'Linear Regression Fit: Actual vs Predicted\n(R^2 = {r2:.4f}, RMSE = {rmse:.4f})', fontsize=12, fontweight='bold')
plt.xlabel('Actual Sentiment Score', fontweight='bold')
plt.ylabel('Predicted Sentiment Score', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../visualizations/linear_regression_fit.png')
plt.show()

--- Model Evaluation Metrics ---
R^2 Score: 0.8949
Mean Absolute Error (MAE): 0.2516
Root Mean Squared Error (RMSE): 0.3099

Feature Coefficients:
  total_messages: +0.5732
  avg_character_length: +0.0027
  avg_word_count: -0.0130
  negative_ratio: -1.0023
  positive_ratio: +1.2214
